run this file first for the first file, then start executing the medallion-notebook via pipeline

In [0]:
%sql
drop table if exists misgauravcatalog.retaildb.orders_bronze;
drop table if exists misgauravcatalog.retaildb.orders_silver;
drop table if exists misgauravcatalog.retaildb.orders_gold;
drop table if exists misgauravcatalog.retaildb.pipeline_watermarks;

In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists misgauravcatalog.retaildb;
use misgauravcatalog.retaildb; 

In [0]:
from pyspark.sql import Row
spark.createDataFrame(
    [Row(
        pipeline_name="bronze_to_silver",
        last_processed_version=-1
    )]

).write \
 .format("delta") \
 .mode("overwrite") \
 .save("gs://databricksfolder/pipelinewatermarks/")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS misgauravcatalog.retaildb.pipeline_watermarks
USING DELTA
LOCATION 'gs://databricksfolder/pipelinewatermarks/'

In [0]:
%sql
SELECT * FROM misgauravcatalog.retaildb.pipeline_watermarks

In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists misgauravcatalog.retaildb;
use misgauravcatalog.retaildb; 

In [0]:
%sql
create table if not exists misgauravcatalog.retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
PARTITIONED BY  (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists misgauravcatalog.retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
# to add _delta_log folder inside the ext location
# unity catalog can create _delta_log folder only in databricks env by default but not for ext location.
empty_silver_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS order_id,
      CAST(NULL AS DATE) AS order_date,
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS order_month,
      CAST(NULL AS TIMESTAMP) AS createdon,
      CAST(NULL AS TIMESTAMP) AS modifiedon
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_silver_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/silver/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
create table if not exists misgauravcatalog.retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
# to add _delta_log folder inside the ext location
# unity catalog can create _delta_log folder only in databricks env by default but not for ext location.
empty_gold_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS num_orders
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_gold_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/gold/")

print("Physical storage log initialized successfully on GCS!")